# Car Sales Data Exploration

Notebook ini digunakan untuk eksplorasi awal dataset, reference data, dan tabel PostgreSQL sebelum logic dipindahkan ke pipeline OOP. Fokus eksplorasi mengikuti requirement task: jumlah baris/kolom, missing values, tipe data, dan sanity check data source.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text


current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in [current_path, *current_path.parents]
    if (path / "dataset").exists()
    and ((path / "pac_car").exists() or (path / "src" / "pac_car").exists())
)
dataset_root = repository_root / "dataset"
brand_path = dataset_root / "car_brand.csv"
state_path = dataset_root / "api_data.json"

brands = pd.read_csv(brand_path, sep="\t")
with state_path.open("r", encoding="utf-8") as file:
    state_payload = json.load(file)
states = pd.DataFrame.from_records(state_payload["regions"])

brands.head(), states.head()

(   brand_car_id brand_name
 0             1      Acura
 1             2       Audi
 2             3    Bentley
 3             4        BMW
 4             5      Buick,
    id_state code        name
 0         1   al     Alabama
 1         2   ak      Alaska
 2         3   az     Arizona
 3         4   ar    Arkansas
 4         5   ca  California)

## Reference Data Profile

In [2]:
reference_summary = pd.DataFrame([
    {
        "source": "brand_csv",
        "rows": len(brands),
        "columns": brands.shape[1],
        "missing_values": int(brands.isna().sum().sum()),
    },
    {
        "source": "state_api_fixture",
        "rows": len(states),
        "columns": states.shape[1],
        "missing_values": int(states.isna().sum().sum()),
    },
])

def series_to_table(series: pd.Series, value_column: str) -> pd.DataFrame:
    table = series.rename(value_column).reset_index()
    table.columns = ["column", value_column]
    return table


display(reference_summary)
display(series_to_table(brands.dtypes.astype(str), "dtype"))
display(series_to_table(states.dtypes.astype(str), "dtype"))
display(series_to_table(brands.isna().sum(), "missing_count"))
display(series_to_table(states.isna().sum(), "missing_count"))

,source,rows,columns,missing_values
0,brand_csv,51,2,0
1,state_api_fixture,68,3,0


,column,dtype
0,brand_car_id,int64
1,brand_name,object


,column,dtype
0,id_state,int64
1,code,object
2,name,object


,column,missing_count
0,brand_car_id,0
1,brand_name,0


,column,missing_count
0,id_state,0
1,code,0
2,name,0


## Docker Project Connection

Cell berikut bisa dipakai dari Jupyter di container Compose ataupun dari host. Jika environment variable Docker tersedia, koneksi memakai service name seperti `source_db`. Jika tidak, koneksi memakai port host dari `.env.example`.

In [3]:
def get_env(key: str, default: str) -> str:
    value = os.getenv(key)
    return value if value not in (None, "") else default


def database_url(prefix: str, default_port: str, database: str) -> str:
    host = get_env(f"{prefix}_POSTGRES_HOST", "localhost")
    port = get_env(f"{prefix}_POSTGRES_PORT", default_port)
    user = get_env(f"{prefix}_POSTGRES_USER", "admin")
    password = get_env(f"{prefix}_POSTGRES_PASSWORD", "admin")
    db_name = get_env(f"{prefix}_POSTGRES_DB", database)
    return f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_name}"


database_targets = {
    "source_sales": (database_url("SRC", "15435", "source_db"), "public.car_sales"),
    "staging_sales": (database_url("STG", "15436", "staging_db"), "public.car_sales"),
    "warehouse_sales": (database_url("WH", "15437", "warehouse_db"), "public.car_sales"),
    "etl_log": (database_url("LOG", "15438", "log_db"), "public.etl_log"),
}

table_counts = []
for label, (url, table_name) in database_targets.items():
    engine = create_engine(url)
    with engine.connect() as connection:
        row_count = connection.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar_one()
    table_counts.append({"table": label, "row_count": row_count})

pd.DataFrame(table_counts)

,table,row_count
0,source_sales,30000
1,staging_sales,30000
2,warehouse_sales,28818
3,etl_log,25


## Source Table Exploration

In [4]:
source_engine = create_engine(database_url("SRC", "15435", "source_db"))
source_sales = pd.read_sql_query(
    """
    SELECT
        id_sales,
        year,
        brand_car,
        transmission,
        state,
        condition,
        odometer,
        color,
        interior,
        mmr,
        sellingprice
    FROM public.car_sales
    ORDER BY id_sales
    """,
    source_engine,
)

source_sales.head()

,id_sales,year,brand_car,transmission,state,condition,odometer,color,interior,mmr,sellingprice
0,1,2014,Chevrolet,automatic,fl,4.0,21507.0,white,black,13450.0,13800.0
1,2,2003,Dodge,,mo,31.0,79712.0,—,black,6025.0,6300.0
2,3,2007,Pontiac,automatic,nj,34.0,65698.0,red,black,7375.0,8000.0
3,4,2011,Toyota,automatic,fl,43.0,23634.0,black,beige,10800.0,11400.0
4,5,2012,Lexus,,pa,35.0,26483.0,black,brown,22500.0,23300.0


In [5]:
source_shape = pd.DataFrame([
    {
        "rows": source_sales.shape[0],
        "columns": source_sales.shape[1],
        "duplicate_id_sales": int(source_sales["id_sales"].duplicated().sum()),
    }
])

source_profile = pd.DataFrame({
    "column": source_sales.columns,
    "dtype": source_sales.dtypes.astype(str).to_numpy(),
    "missing_count": source_sales.isna().sum().to_numpy(),
    "missing_percentage": (source_sales.isna().mean() * 100).round(2).to_numpy(),
    "unique_count": source_sales.nunique(dropna=True).to_numpy(),
})

display(source_shape)
display(source_profile)

,rows,columns,duplicate_id_sales
0,30000,11,0


,column,dtype,missing_count,missing_percentage,unique_count
0,id_sales,int64,0,0.00,30000
1,year,int64,0,0.00,29
2,brand_car,object,0,0.00,52
3,transmission,object,0,0.00,4
4,state,object,0,0.00,39
5,condition,float64,659,2.20,41
6,odometer,float64,7,0.02,26930
7,color,object,0,0.00,22
8,interior,object,0,0.00,17
9,mmr,float64,2,0.01,1002


In [6]:
numeric_columns = ["year", "condition", "odometer", "mmr", "sellingprice"]
source_sales[numeric_columns].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
year,30000.0,2010.04,3.98,1985.0,2007.0,2012.0,2013.0,2015.0
condition,29341.0,30.77,13.42,1.0,24.0,35.0,42.0,49.0
odometer,29993.0,68251.49,53430.66,1.0,27926.0,51923.0,99277.0,999999.0
mmr,29998.0,13750.14,9690.64,25.0,7050.0,12300.0,18300.0,146000.0
sellingprice,30000.0,13587.19,9729.46,100.0,6900.0,12200.0,18200.0,154000.0


In [7]:
categorical_columns = ["brand_car", "transmission", "state", "color", "interior"]
categorical_profile = []
for column in categorical_columns:
    value_counts = source_sales[column].value_counts(dropna=False)
    top_value = value_counts.index[0] if not value_counts.empty else None
    top_frequency = int(value_counts.iloc[0]) if not value_counts.empty else 0
    categorical_profile.append({
        "column": column,
        "unique_count": source_sales[column].nunique(dropna=True),
        "top_value": top_value,
        "top_frequency": top_frequency,
    })

pd.DataFrame(categorical_profile)

,column,unique_count,top_value,top_frequency
0,brand_car,52,Ford,5098
1,transmission,4,automatic,25512
2,state,39,fl,4530
3,color,22,black,5953
4,interior,17,black,12989


Notebook ini bisa dibuka dari service Docker `notebook` di `http://localhost:18888`. Dari host, koneksi database memakai port `15435-15438`; dari container, koneksi memakai hostname service Compose seperti `source_db:5432`.